In [133]:
import pandas as pd
import numpy as np

In [134]:
transaction_data = pd.read_csv("../data/transaction_data.csv")
meta_data = pd.read_csv("../data/meta_data.csv")

## Data Cleaning (from Assignment 1 feedback)

In [135]:
transaction_data_clean = transaction_data
transaction_data_clean.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]
transaction_data_clean

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,FA6E03CE,WIP,MTO,28,1006.099483,PL-AC,B3B6,2023-01-01,98
1,3712740E,FG,MTS,14,53.177793,NaN,E276,2023-01-01,16
2,499E95CC,WIP,MTS,28,990.435666,PL-AC,3A59,2023-01-01,79
3,49802640,FG,MTS,28,1009.209954,JP-CC,CE04,2023-01-01,101
4,AB8250FB,FG,MTS,28,1028.199254,JP-CC,14EF,2023-01-01,118
...,...,...,...,...,...,...,...,...,...
799032,B30BB0A8,WIP,MTO,21,206.954154,NaN,3190,2025-12-30,40
799033,692C5C88,FG,MTS,14,46.534071,CA-BB,14EF,2025-12-30,10
799034,79556710,WIP,MTS,28,992.249632,PL-AC,7A50,2025-12-30,94
799035,4C00E2C8,FG,MTS,28,999.806121,CA-BB,B8FD,2025-12-30,88


In [136]:
transaction_data_clean.loc[transaction_data_clean['lead_time'] > 28, 'lead_time'] /= 100

Here we looked for all instances where ```lead_time``` is greater than 28 (all are between 2100-2800), and assumed that they were just typos. So we divided them by 100 to get them to par with the other values.

In [137]:
transaction_data_clean.loc[transaction_data_clean['order_quantity'] < 0, 'order_quantity'] = transaction_data_clean['order_quantity'].abs()
transaction_data_clean.loc[transaction_data_clean['unit_price'] < 0, 'unit_price'] = transaction_data_clean['unit_price'].abs()

We also assumed that negative values in ```order_quantity``` and ```unit_price``` were typos, so we converted them to positive by taking the absolute value for all negative instances.

In [138]:
transaction_data_clean.isna().sum()

sku_number               187
inventory_type         15496
stocking_type          18401
lead_time                  0
unit_price                 0
manufacturing_site    155047
division_code          56592
transaction_date           0
order_quantity             0
dtype: int64

In [139]:
categorical_cols = ['sku_number', 'inventory_type', 'stocking_type', 'manufacturing_site', 'division_code']
transaction_data_clean[categorical_cols] = transaction_data_clean[categorical_cols].fillna('unknown')

Instead of removing all rows in NaN values, we changed them to 'unknown'.

In [140]:
transaction_data_clean.isna().sum()

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

In [141]:
transaction_data_clean['country'] = None

transaction_data_clean.loc[transaction_data_clean['manufacturing_site'].str.startswith(
    'PL', na=False), 'country'] = 'Poland'
transaction_data_clean.loc[transaction_data_clean['manufacturing_site'].str.startswith(
    'JP', na=False), 'country'] = 'Japan'
transaction_data_clean.loc[transaction_data_clean['manufacturing_site'].str.startswith(
    'US', na=False), 'country'] = 'United States'
transaction_data_clean.loc[transaction_data_clean['manufacturing_site'].str.startswith(
    'CA', na=False), 'country'] = 'Canada'

transaction_data_clean

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
0,FA6E03CE,WIP,MTO,28,1006.099483,PL-AC,B3B6,2023-01-01,98,Poland
1,3712740E,FG,MTS,14,53.177793,unknown,E276,2023-01-01,16,None
2,499E95CC,WIP,MTS,28,990.435666,PL-AC,3A59,2023-01-01,79,Poland
3,49802640,FG,MTS,28,1009.209954,JP-CC,CE04,2023-01-01,101,Japan
4,AB8250FB,FG,MTS,28,1028.199254,JP-CC,14EF,2023-01-01,118,Japan
...,...,...,...,...,...,...,...,...,...,...
799032,B30BB0A8,WIP,MTO,21,206.954154,unknown,3190,2025-12-30,40,None
799033,692C5C88,FG,MTS,14,46.534071,CA-BB,14EF,2025-12-30,10,Canada
799034,79556710,WIP,MTS,28,992.249632,PL-AC,7A50,2025-12-30,94,Poland
799035,4C00E2C8,FG,MTS,28,999.806121,CA-BB,B8FD,2025-12-30,88,Canada


We included the full country names, instead of abbreviating them.

## Question 1 — Required Transformation

### Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.

In [142]:
filtered_transaction_data_clean = transaction_data_clean[(transaction_data_clean['sku_number'] != 'unknown') & (transaction_data_clean['stocking_type'] == 'MTS')]

sku_transformed_dataset = filtered_transaction_data_clean.groupby('sku_number').agg(
    min_quantity=('order_quantity', 'min'),
    max_quantity=('order_quantity', 'max'),
    avg_quantity=('order_quantity', 'mean'),
    median_quantity=('order_quantity', 'median'),
    var_quantity=('order_quantity', 'var'),
    std_quantity=('order_quantity', 'std'),
    avg_lead_time=('lead_time', 'mean')
    ).reset_index()

sku_transformed_dataset


,sku_number,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time
0,000B9F05,0,31,8.402062,7.0,35.826246,5.985503,14.000000
1,00ED763A,38,170,100.317410,101.0,394.542851,19.863103,28.000000
2,01061156,0,22,7.618182,6.0,28.366639,5.326034,14.000000
3,0111AC2E,0,29,9.193370,8.0,37.512400,6.124737,14.000000
4,01337338,0,32,9.282353,9.0,38.490756,6.204092,14.000000
...,...,...,...,...,...,...,...,...
481,FEBCC979,29,168,99.774029,100.0,395.635747,19.890594,28.000000
482,FF1E8FB2,0,79,15.996359,12.5,173.622101,13.176574,20.074029
483,FF35F9DC,0,164,91.150387,96.0,1049.415689,32.394686,28.000000
484,FF658282,0,38,9.242105,8.0,58.334378,7.637695,14.000000


First, we filtered out all rows where ```sku_number``` is unknown, and kept only the rows where ```stocking_type``` is set to 'MTS'. Then we aggregated with the specified statistics. For each SKU, we calculated the minimum, maximum, average, median, variance, and standard deviation of order quantity, along with the average lead time. This resulted in 486 unique SKU numbers.

## Question 2 — Safety Stock Calculation

### Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. You will need to find the z-score of the given service level coefficients, for which you can use the ppf function of SciPy.Stat module of the SciPy package, or simply use the numeric values from any standard normal distribution function.

In [143]:
variability_factor = sku_transformed_dataset['std_quantity'] * np.sqrt(sku_transformed_dataset['avg_lead_time'])

sku_transformed_dataset['ss_75'] = 0.6745 * variability_factor
sku_transformed_dataset['ss_90'] = 1.2816 * variability_factor
sku_transformed_dataset['ss_95'] = 1.6449 * variability_factor

sku_transformed_dataset


,sku_number,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,ss_75,ss_90,ss_95
0,000B9F05,0,31,8.402062,7.0,35.826246,5.985503,14.000000,15.105901,28.702331,36.838689
1,00ED763A,38,170,100.317410,101.0,394.542851,19.863103,28.000000,70.893768,134.703414,172.888301
2,01061156,0,22,7.618182,6.0,28.366639,5.326034,14.000000,13.441567,25.539974,32.779887
3,0111AC2E,0,29,9.193370,8.0,37.512400,6.124737,14.000000,15.457292,29.370000,37.695625
4,01337338,0,32,9.282353,9.0,38.490756,6.204092,14.000000,15.657564,29.750532,38.184028
...,...,...,...,...,...,...,...,...,...,...,...
481,FEBCC979,29,168,99.774029,100.0,395.635747,19.890594,28.000000,70.991889,134.889852,173.127588
482,FF1E8FB2,0,79,15.996359,12.5,173.622101,13.176574,20.074029,39.820044,75.661035,97.108955
483,FF35F9DC,0,164,91.150387,96.0,1049.415689,32.394686,28.000000,115.620474,219.687472,281.963110
484,FF658282,0,38,9.242105,8.0,58.334378,7.637695,14.000000,19.275616,36.625099,47.007354


We calculated the safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. We found the z-scores by inputting numeric values from standard normal distribution functions. We broke down the formula from the assignment instructions into 2 separate steps.

## Question 3 — Safety Stock Distribution

## Now that we have the safety stock values, write appropriate code to answer the following questions. For this question, you may focus on the 95% service level coefficient.

### Which SKU has the largest safety stock?

In [144]:
top_sku_ss = sku_transformed_dataset.sort_values(by = 'ss_95', ascending = False).iloc[0]
top_sku_ss

sku_number            E46FB532
min_quantity                 0
max_quantity               168
avg_quantity         69.349804
median_quantity           85.0
var_quantity       1990.740324
std_quantity         44.617713
avg_lead_time        24.767508
ss_75               149.771927
ss_90               284.577763
ss_95               365.248098
Name: 436, dtype: object

We sorted ```ss_95``` from largest to smallest and used iloc to find the value of the first row.

### Which SKU has the smallest safety stock?

In [145]:
bottom_sku_ss = sku_transformed_dataset.sort_values(by = 'ss_95', ascending = True).iloc[0]
bottom_sku_ss

sku_number          5971C01E
min_quantity               0
max_quantity              22
avg_quantity             7.2
median_quantity          7.0
var_quantity       27.209524
std_quantity        5.216275
avg_lead_time           14.0
ss_75              13.164563
ss_90              25.013645
ss_95              32.104358
Name: 181, dtype: object

We sorted ```ss_95``` from smallest to largest and used iloc to find the value of the first row.

### What is the average safety stock value across all SKUs?

In [146]:
avg_ss_95 = sku_transformed_dataset['ss_95'].mean()
avg_ss_95

np.float64(119.84611206796471)

We directly applied ```mean()``` to the ```ss_95``` column.

## References

1. Databricks. (2020, April 22). "How a Fresh Approach to Safety Stock Analysis Can Optimize Inventory." Retrieved November 20th, 2026. https://www.databricks.com/blog/2020/04/22/how-a-fresh-approach-to-safety-stock-analysis-can-optimize-inventory.html
2. SciPy. (n.d.). "scipy.stats.mstats.mquantiles." Retrieved November 20th, 2026. https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mstats.mquantiles.html

## Side Quest

The above safety stock calculation formula is one of the most popular approaches in practice, but it relies on a rather strong assumption that the underlying demand is normally distributed. A quick visualization of many SKUs reveals a non-normal distribution for many of them, which poses some risk.

You can receive a 15% extra point on this assignment if, in addition to the above formula, you also calculate the safety stock values using a simple non-parametric quantile function.

Note 1: I strongly suggest researching quantiles and empirical quantiles to learn more about them before making your calculation.

Note 2: The reason the simple quantile function is referred to as non-parametric is that we do not assume the demand follows the parametric normal distribution.

Note 3: The simplest approach to calculate specific empirical quantities in Python is using the mquantiles function of the scipy.stats.mstats module in the scipy package.

In [147]:
from scipy.stats.mstats import mquantiles


In [148]:
sku_transformed_dataset = filtered_transaction_data_clean.groupby('sku_number').agg(
    avg_quantity = ('order_quantity', 'mean'),
    avg_lead_time = ('lead_time', 'mean'),
    q_95 = ('order_quantity', lambda x: mquantiles(x, prob = 0.95)[0])
).reset_index()

sqrt_lead_time = np.sqrt(sku_transformed_dataset['avg_lead_time'])

sku_transformed_dataset['np_ss_95'] = (sku_transformed_dataset['q_95'] - sku_transformed_dataset['avg_quantity']) * sqrt_lead_time

We grouped our transaction data by SKU to calculate the average order quantity, average lead time, and the 95th percentile of historical order quantities. We then calculated our non-parametric safety stock by subtracting the average order quantity from the 95th percentile and multiplying the difference by the square root of average lead time.

In [149]:
bottom_sku_ss = sku_transformed_dataset.sort_values(by = 'np_ss_95', ascending = True).iloc[0]
bottom_sku_ss

sku_number        4A6D52B8
avg_quantity      7.649485
avg_lead_time         14.0
q_95                 15.74
np_ss_95         30.271937
Name: 156, dtype: object

In [150]:
top_sku_ss = sku_transformed_dataset.sort_values(by = 'np_ss_95', ascending = False).iloc[0]
top_sku_ss


sku_number         E46FB532
avg_quantity      69.349804
avg_lead_time     24.767508
q_95                 129.94
np_ss_95         301.539014
Name: 436, dtype: object

In [151]:
sku_transformed_dataset['np_ss_95'].mean()

np.float64(112.02121379142396)

Then, like we did for Problem #3, we calculated the SKUs with the largest and smallest safety stocks (for the 95th percentile), as well as the average safety stock value across all SKUs. As expected, when comparing these values to the values from Problem #3, we see some differences: the mean value is slightly lower and the SKU number for smallest safety stock is different. For the largest safety stock, the SKU number is the same for both.